# Project 1 — Baseline Analysis: FB-DDPG vs DIAYN
**EECS 567 — Reinforcement Learning**

### Structure
1. Mount Drive & cd into repo
2. Install system dependencies
3. *(Commented)* Pretraining commands
4. Configuration & imports
5. Helper functions
6. Load pretraining eval data (`eval.csv`)
7. Inspect `test_rewards.json` (post fine-tuning)
8. **Figure 1** — Pretraining learning curves (two-panel)
9. **Figure 2** — FB vs DIAYN on walker_walk (head-to-head)
10. **Figure 3** — Pretraining final performance bar chart
11. **Figure 4** — Post fine-tuning bar chart (from `test_rewards.json`)
12. Summary stats tables
13. Download figures

## 1. Mount Drive & cd into repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/Colab Notebooks/controllable_agent"

## 2. Install System Dependencies for MuJoCo

In [ ]:
!apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev patchelf

## 3. Pretraining Commands *(commented — already done)*
Uncomment only if retraining from scratch.

In [ ]:
# ── FB-DDPG ───────────────────────────────────────────────────────────────────

# FB / walker_walk
# !python -m url_benchmark.pretrain agent=fb_ddpg task=walker_walk seed=1 use_tb=1
# !python -m url_benchmark.pretrain agent=fb_ddpg task=walker_walk seed=2 use_tb=1
# !python -m url_benchmark.pretrain agent=fb_ddpg task=walker_walk seed=3 use_tb=1

# FB / quadruped_walk
# !python -m url_benchmark.pretrain agent=fb_ddpg task=quadruped_walk seed=1 use_tb=1
# !python -m url_benchmark.pretrain agent=fb_ddpg task=quadruped_walk seed=2 use_tb=1
# !python -m url_benchmark.pretrain agent=fb_ddpg task=quadruped_walk seed=3 use_tb=1

# ── DIAYN ─────────────────────────────────────────────────────────────────────

# DIAYN / walker_walk
# !python -m url_benchmark.pretrain agent=diayn task=walker_walk seed=1 use_tb=1
# !python -m url_benchmark.pretrain agent=diayn task=walker_walk seed=2 use_tb=1
# !python -m url_benchmark.pretrain agent=diayn task=walker_walk seed=3 use_tb=1

# DIAYN / cheetah_run
# !python -m url_benchmark.pretrain agent=diayn task=cheetah_run seed=1 use_tb=1
# !python -m url_benchmark.pretrain agent=diayn task=cheetah_run seed=2 use_tb=1
# !python -m url_benchmark.pretrain agent=diayn task=cheetah_run seed=3 use_tb=1

print("Pretraining commands are commented out.")

## 4. Configuration & Imports

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

# ── UPDATE IF NEEDED ──────────────────────────────────────────────────────────
RUNS_DIR = "/content/drive/MyDrive/Project 1 Evals/runs"
# ─────────────────────────────────────────────────────────────────────────────

FIG_DIR = "/content/figures"
os.makedirs(FIG_DIR, exist_ok=True)

RUNS = {
    "FB / walker_walk": {
        "agent":   "fb_ddpg",
        "task":    "walker_walk",
        "domain":  "walker",
        "folders": [
            "215606_fb_ddpg_walker_walk_online",
            "185625_fb_ddpg_walker_walk_online",
            "204104_fb_ddpg_walker_walk_online",
        ],
        "color":     "#2196F3",
        "linestyle": "-",
    },
    "FB / quadruped_walk": {
        "agent":   "fb_ddpg",
        "task":    "quadruped_walk",
        "domain":  "quadruped",
        "folders": [
            "060149_fb_ddpg_quadruped_walk_online",
            "092501_fb_ddpg_quadruped_walk_online",
            "122214_fb_ddpg_quadruped_walk_online",
        ],
        "color":     "#4CAF50",
        "linestyle": "-",
    },
    "DIAYN / walker_walk": {
        "agent":   "diayn",
        "task":    "walker_walk",
        "domain":  "walker",
        "folders": [
            "165350_diayn_walker_walk_online",
            "040441_diayn_walker_walk_online",
            "175202_diayn_walker_walk_online",
        ],
        "color":     "#FF5722",
        "linestyle": "--",
    },
    "DIAYN / cheetah_run": {
        "agent":   "diayn",
        "task":    "cheetah_run",
        "domain":  "cheetah",
        "folders": [
            "121039_diayn_cheetah_run_online",
            "163347_diayn_cheetah_run_online",
            "091356_diayn_cheetah_run_online",
        ],
        "color":     "#9C27B0",
        "linestyle": "--",
    },
}

plt.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "sans-serif",
    "axes.labelsize":    11,
})

# Verify path
assert os.path.exists(RUNS_DIR), f"RUNS_DIR not found: {RUNS_DIR}"
print("Folders in RUNS_DIR:")
for f in sorted(os.listdir(RUNS_DIR)):
    print(" ", f)

## 5. Helper Functions

In [ ]:
def load_eval_csvs(folders: list, runs_dir: str) -> list:
    """Load eval.csv from each run folder."""
    dfs = []
    for folder in folders:
        path = Path(runs_dir) / folder / "eval.csv"
        if not path.exists():
            print(f"  [WARNING] Not found: {path}")
            continue
        df = pd.read_csv(path)
        dfs.append(df)
        print(f"  Loaded {folder}  ({len(df)} rows)")
    return dfs


def load_test_rewards(folders: list, runs_dir: str) -> list:
    """Load test_rewards.json from each run folder."""
    results = []
    for folder in folders:
        path = Path(runs_dir) / folder / "test_rewards.json"
        if not path.exists():
            print(f"  [WARNING] Not found: {path}")
            continue
        with open(path) as f:
            data = json.load(f)
        results.append(data)
        print(f"  Loaded {folder}")
    return results


def smooth(values: np.ndarray, window: int = 7) -> np.ndarray:
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="same")


def align_and_aggregate(dfs: list, x_col="frame", y_col="episode_reward"):
    """Interpolate seeds onto a common x grid. Returns (x, mean, std)."""
    if not dfs:
        return None, None, None
    all_x = [df[x_col].values for df in dfs]
    x_min = max(a.min() for a in all_x)
    x_max = min(a.max() for a in all_x)
    x_common = np.linspace(x_min, x_max, max(len(a) for a in all_x))
    interped = [
        np.interp(x_common, df[x_col].values, df[y_col].values)
        for df in dfs
    ]
    stacked = np.array(interped)
    return x_common, stacked.mean(axis=0), stacked.std(axis=0)


def final_mean_csv(df: pd.DataFrame, last_n_pct: float = 0.1) -> float:
    """Mean episode_reward over last fraction of a pretraining eval.csv."""
    y = df["episode_reward"].values
    n = max(1, int(len(y) * last_n_pct))
    return float(y[-n:].mean())


def task_mean_from_json(data: dict, task_key: str) -> float | None:
    """
    Average reward for a specific task from test_rewards.json.
    test_rewards.json format: {task_name: [ep_reward_1, ep_reward_2, ...]}
    Falls back to first key if exact task_key not found.
    """
    vals = data.get(task_key)
    if vals is None:
        # Try partial match (e.g. key is 'walker_walk' but stored as 'rewards')
        for k, v in data.items():
            if task_key in k or k in task_key:
                vals = v
                break
    if vals is None:
        return None
    return float(sum(vals) / len(vals))


print("Helper functions loaded.")

## 6. Load All Data

In [ ]:
pretrain_data = {}   # label -> list of eval.csv DataFrames
test_rewards  = {}   # label -> list of test_rewards.json dicts

for label, cfg in RUNS.items():
    print(f"\n── {label}")
    pretrain_data[label] = load_eval_csvs(cfg["folders"], RUNS_DIR)
    test_rewards[label]  = load_test_rewards(cfg["folders"], RUNS_DIR)

print("\nAll data loaded.")

## 7. Inspect test_rewards.json
This shows all the downstream tasks `pretrain.py` evaluated at the end of training.

In [ ]:
# Print full contents for one run of each agent to see all tasks available
for label, cfg in RUNS.items():
    runs = test_rewards[label]
    if not runs:
        print(f"{label}: no test_rewards.json found")
        continue
    print(f"\n── {label}")
    data = runs[0]   # first seed
    for task, vals in data.items():
        mean = round(sum(vals) / len(vals), 2)
        print(f"  {task:<35} mean={mean:.2f}  (n={len(vals)} episodes)")

## 8. Figure 1 — Pretraining Learning Curves
Two-panel: walker-scale on the left, FB/quadruped with individual seeds on the right.
X-axis starts at 100k to skip the noisy early warmup phase.

In [ ]:
WARMUP = 100_000

fig = plt.figure(figsize=(13, 5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[2, 1], wspace=0.35)
ax_left  = fig.add_subplot(gs[0])
ax_right = fig.add_subplot(gs[1])

# ── Left panel: walker + cheetah ──────────────────────────────────────────────
for label in ["FB / walker_walk", "DIAYN / walker_walk", "DIAYN / cheetah_run"]:
    cfg = RUNS[label]
    x, mean, std = align_and_aggregate(pretrain_data[label])
    if x is None:
        continue
    mask = x >= WARMUP
    x, mean, std = x[mask], mean[mask], std[mask]
    ax_left.plot(x, smooth(mean), label=label,
                 color=cfg["color"], linestyle=cfg["linestyle"], linewidth=2)
    ax_left.fill_between(x, smooth(mean - std), smooth(mean + std),
                         alpha=0.15, color=cfg["color"])

ax_left.set_title("Walker & Cheetah — Pretraining Eval\n(after 100k warmup)",
                  fontweight="bold")
ax_left.set_xlabel("Environment Frames")
ax_left.set_ylabel("Episode Reward (eval)")
ax_left.legend(fontsize=9)
ax_left.grid(True, alpha=0.25)
ax_left.set_xlim(left=WARMUP)
ax_left.set_ylim(bottom=0)

# ── Right panel: FB/quadruped with individual seeds ───────────────────────────
cfg = RUNS["FB / quadruped_walk"]
dfs = pretrain_data["FB / quadruped_walk"]
x, mean, std = align_and_aggregate(dfs)
if x is not None:
    mask = x >= WARMUP
    x_m, mean_m, std_m = x[mask], mean[mask], std[mask]
    # Individual seeds (faint)
    for df in dfs:
        xi = df["frame"].values
        yi = df["episode_reward"].values
        mi = xi >= WARMUP
        ax_right.plot(xi[mi], yi[mi], color=cfg["color"], alpha=0.25, linewidth=1)
    # Mean
    ax_right.plot(x_m, smooth(mean_m), color=cfg["color"], linewidth=2.5,
                  label="Mean (3 seeds)")
    ax_right.fill_between(x_m, smooth(mean_m - std_m), smooth(mean_m + std_m),
                          alpha=0.2, color=cfg["color"])

ax_right.set_title("FB / quadruped_walk\n(individual seeds — high variance)",
                   fontweight="bold")
ax_right.set_xlabel("Environment Frames")
ax_right.set_ylabel("Episode Reward (eval)")
ax_right.legend(fontsize=8)
ax_right.grid(True, alpha=0.25)
ax_right.set_xlim(left=WARMUP)
ax_right.set_ylim(bottom=0)

fig.suptitle("Pretraining Learning Curves — FB-DDPG vs DIAYN",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig1_pretrain_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig1_pretrain_curves.png")

## 9. Figure 2 — FB vs DIAYN on Walker Walk (Head-to-Head)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

for label in ["FB / walker_walk", "DIAYN / walker_walk"]:
    cfg = RUNS[label]
    x, mean, std = align_and_aggregate(pretrain_data[label])
    if x is None:
        continue
    mask = x >= WARMUP
    x, mean, std = x[mask], mean[mask], std[mask]
    n = len(pretrain_data[label])
    ax.plot(x, smooth(mean), label=f"{label}  (n={n})",
            color=cfg["color"], linestyle=cfg["linestyle"], linewidth=2.5)
    ax.fill_between(x, smooth(mean - std), smooth(mean + std),
                    alpha=0.2, color=cfg["color"])
    # Annotate final value
    final_val = smooth(mean)[-1]
    ax.annotate(f"{final_val:.0f}",
                xy=(x[-1], final_val),
                xytext=(8, 0), textcoords="offset points",
                color=cfg["color"], fontsize=10, fontweight="bold", va="center")

ax.set_title("FB-DDPG vs DIAYN — Walker Walk (Direct Comparison)\n"
             "Pretraining eval, after 100k warmup", fontweight="bold")
ax.set_xlabel("Environment Frames")
ax.set_ylabel("Episode Reward (eval)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
ax.set_xlim(left=WARMUP)
ax.set_ylim(bottom=0)
ax.text(0.02, 0.05,
        "Note: rewards during pretraining — not post fine-tuning",
        transform=ax.transAxes, fontsize=8, color="gray", va="bottom")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig2_walker_head_to_head.png", dpi=150)
plt.show()
print("Saved: fig2_walker_head_to_head.png")

## 10. Figure 3 — Pretraining Final Performance Bar Chart

In [ ]:
rng = np.random.default_rng(42)
labels, means, stds, colors, all_seeds = [], [], [], [], []

for label, cfg in RUNS.items():
    dfs = pretrain_data[label]
    labels.append(label)
    colors.append(cfg["color"])
    if dfs:
        sv = [final_mean_csv(df) for df in dfs]
        means.append(float(np.mean(sv)))
        stds.append(float(np.std(sv)))
        all_seeds.append(sv)
    else:
        means.append(0.0); stds.append(0.0); all_seeds.append([])

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.bar(x, means, yerr=stds, capsize=6, color=colors,
              width=0.55, edgecolor="black", linewidth=0.8, alpha=0.65)

for i, (seeds, col) in enumerate(zip(all_seeds, colors)):
    jitter = rng.uniform(-0.08, 0.08, len(seeds))
    ax.scatter(x[i] + jitter, seeds, color=col, s=65, zorder=5,
               edgecolors="white", linewidths=0.8)

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + s + max(means) * 0.015,
            f"{m:.1f}", ha="center", va="bottom",
            fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Mean Episode Reward (last 10% of pretraining)")
ax.set_title("Pretraining Final Performance — Mean ± Std with Individual Seeds",
             fontsize=12, fontweight="bold")
ax.grid(True, axis="y", alpha=0.25)
ax.set_ylim(bottom=0)
ax.legend(handles=[
    Line2D([0],[0], marker='o', color='gray', linestyle='None',
           markersize=7, label='Individual seed'),
    plt.Rectangle((0,0), 1, 1, fc='gray', alpha=0.65, label='Mean ± Std'),
], fontsize=9, loc="upper left")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig3_pretrain_bars.png", dpi=150)
plt.show()
print("Saved: fig3_pretrain_bars.png")

## 11. Figure 4 — Post Fine-tuning Performance (from test_rewards.json)

`test_rewards.json` is written by `pretrain.py`'s `finalize()` method at the end of training.
It runs the pretrained agent on all downstream tasks for the domain and records episode rewards.
This is the post-fine-tuning metric comparable to paper results.

> **Run cell 7 first** to confirm the exact task key names stored in your json files,
> then update `task_key` below if needed.

In [ ]:
# ── Collect per-seed means from test_rewards.json ─────────────────────────────
ft_labels, ft_means, ft_stds, ft_colors, ft_seeds = [], [], [], [], []
rng2 = np.random.default_rng(99)

for label, cfg in RUNS.items():
    runs = test_rewards[label]
    ft_labels.append(label)
    ft_colors.append(cfg["color"])

    if not runs:
        ft_means.append(0.0); ft_stds.append(0.0); ft_seeds.append([])
        continue

    task_key = cfg["task"]   # e.g. "walker_walk"
    seed_vals = []
    for run_data in runs:
        m = task_mean_from_json(run_data, task_key)
        if m is not None:
            seed_vals.append(m)

    if seed_vals:
        ft_means.append(float(np.mean(seed_vals)))
        ft_stds.append(float(np.std(seed_vals)))
        ft_seeds.append(seed_vals)
    else:
        ft_means.append(0.0); ft_stds.append(0.0); ft_seeds.append([])

# ── Plot ──────────────────────────────────────────────────────────────────────
x = np.arange(len(ft_labels))
fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.bar(x, ft_means, yerr=ft_stds, capsize=6, color=ft_colors,
              width=0.55, edgecolor="black", linewidth=0.8, alpha=0.65)

for i, (seeds, col) in enumerate(zip(ft_seeds, ft_colors)):
    jitter = rng2.uniform(-0.08, 0.08, len(seeds))
    ax.scatter(x[i] + jitter, seeds, color=col, s=65, zorder=5,
               edgecolors="white", linewidths=0.8)

max_val = max(ft_means) if any(ft_means) else 1
for bar, m, s in zip(bars, ft_means, ft_stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + s + max_val * 0.015,
            f"{m:.1f}", ha="center", va="bottom",
            fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(ft_labels, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Mean Episode Reward (post fine-tuning)")
ax.set_title("Post Fine-tuning Performance (test_rewards.json)\nMean ± Std with Individual Seeds",
             fontsize=12, fontweight="bold")
ax.grid(True, axis="y", alpha=0.25)
ax.set_ylim(bottom=0)
ax.legend(handles=[
    Line2D([0],[0], marker='o', color='gray', linestyle='None',
           markersize=7, label='Individual seed'),
    plt.Rectangle((0,0), 1, 1, fc='gray', alpha=0.65, label='Mean ± Std'),
], fontsize=9, loc="upper left")

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig4_finetune_bars.png", dpi=150)
plt.show()
print("Saved: fig4_finetune_bars.png")

## 12. Summary Stats Tables

In [ ]:
# ── Table 1: Pretraining eval (eval.csv) ─────────────────────────────────────
print("Table 1: Pretraining Eval Performance")
rows = []
for label, cfg in RUNS.items():
    dfs = pretrain_data[label]
    if dfs:
        sv = [final_mean_csv(df) for df in dfs]
        peak = max(df["episode_reward"].max() for df in dfs)
        rows.append({
            "Run":              label,
            "Seeds":            len(sv),
            "Final Mean":       round(float(np.mean(sv)), 2),
            "Final Std":        round(float(np.std(sv)), 2),
            "Peak (any seed)":  round(float(peak), 2),
        })
    else:
        rows.append({"Run": label, "Seeds": 0,
                     "Final Mean": "N/A", "Final Std": "N/A",
                     "Peak (any seed)": "N/A"})
display(pd.DataFrame(rows).set_index("Run"))

In [ ]:
# ── Table 2: Post fine-tuning (test_rewards.json) ─────────────────────────────
print("Table 2: Post Fine-tuning Performance (test_rewards.json)")
rows = []
for label, cfg in RUNS.items():
    runs = test_rewards[label]
    if runs:
        task_key = cfg["task"]
        sv = [task_mean_from_json(r, task_key) for r in runs]
        sv = [v for v in sv if v is not None]
        rows.append({
            "Run":           label,
            "Task":          task_key,
            "Seeds":         len(sv),
            "Mean Reward":   round(float(np.mean(sv)), 2) if sv else "N/A",
            "Std":           round(float(np.std(sv)), 2)  if sv else "N/A",
            "Per-seed":      [round(v, 1) for v in sv],
        })
    else:
        rows.append({"Run": label, "Task": cfg["task"], "Seeds": 0,
                     "Mean Reward": "N/A", "Std": "N/A", "Per-seed": []})
display(pd.DataFrame(rows).set_index("Run"))

In [ ]:
# ── Table 3: All downstream tasks from test_rewards.json ─────────────────────
# This shows every task pretrain.py evaluated, not just the primary one.
print("Table 3: All downstream tasks in test_rewards.json (mean across seeds)")
rows = []
for label, cfg in RUNS.items():
    runs = test_rewards[label]
    if not runs:
        continue
    # Collect all task keys across all seeds
    all_tasks = set(k for r in runs for k in r.keys())
    for task in sorted(all_tasks):
        sv = []
        for r in runs:
            if task in r:
                sv.append(float(sum(r[task]) / len(r[task])))
        rows.append({
            "Agent/Pretrain Task": label,
            "Eval Task":           task,
            "Seeds":               len(sv),
            "Mean Reward":         round(float(np.mean(sv)), 2) if sv else "N/A",
            "Std":                 round(float(np.std(sv)), 2)  if sv else "N/A",
        })
display(pd.DataFrame(rows).set_index(["Agent/Pretrain Task", "Eval Task"]))

## 13. Download All Figures

In [ ]:
from google.colab import files

for fname in [
    "fig1_pretrain_curves.png",
    "fig2_walker_head_to_head.png",
    "fig3_pretrain_bars.png",
    "fig4_finetune_bars.png",
]:
    fpath = f"{FIG_DIR}/{fname}"
    if os.path.exists(fpath):
        files.download(fpath)
        print(f"Downloading {fname}")
    else:
        print(f"[SKIP] {fname} — run its figure cell first")